In [ ]:
import os
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv
import json
from pathlib import Path
from bs4 import BeautifulSoup

import pandas as pd
import numpy as np

import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

import random

In [ ]:
load_dotenv()

CLIENT_ID = os.getenv("IGDB_CLIENT_ID")
CLIENT_SECRET = os.getenv("IGDB_CLIENT_SECRET")

IGDB_BASE_URL = "https://api.igdb.com/v4"

In [ ]:
def get_access_token():

    url = "https://id.twitch.tv/oauth2/token"

    params = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials"
    }

    response = requests.post(url, params=params)

    response.raise_for_status()

    data = response.json()

    return data["access_token"]

In [ ]:
def get_games(limit=500, offset=0):

    access_token = get_access_token()

    url = f"{IGDB_BASE_URL}/games"

    headers = {
        "Client-ID": CLIENT_ID,
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }

    query = f"""
        fields
            id,
            name,
            slug,
            summary,
            first_release_date,
            game_type,
            rating,
            rating_count,
            total_rating,
            total_rating_count,
            aggregated_rating,
            aggregated_rating_count,
            genres,
            platforms,
            involved_companies,
            game_modes,
            themes,
            player_perspectives,
            keywords,
            franchises,
            collections,
            expansions,
            dlcs,
            hypes;
        
        limit {limit};
        offset {offset};
    """

    # where first_release_date >= 1262304000 --> desde 2010 aproximadamente

    response = requests.post(
        url,
        headers=headers,
        data=query
    )

    if response.status_code != 200:
        print("Status code:", response.status_code)
        print("Respuesta de IGDB:")
        print(response.text)

    response.raise_for_status()

    return response.json()

In [ ]:
# Prueba con offset para 5000 juegos

all_games = []

for offset in range(0, 5000, 500):

    print(f"Descargando juegos {offset} - {offset + 500}")

    games = get_games(
        limit=500,
        offset=offset
    )

    all_games.extend(games)

    time.sleep(0.3)

In [ ]:
df_games = pd.DataFrame(all_games)

df_games.head()

In [ ]:
df_games.columns.tolist()

In [ ]:
df_games.info()

In [ ]:
df_games.isnull().sum()

In [ ]:
df_games["first_release_date"] = pd.to_datetime(
    df_games["first_release_date"],
    unit="s",
    errors="coerce"
)

In [ ]:
df_games["first_release_date"].describe()

In [ ]:
df_games["game_type"].value_counts(dropna=False)

In [ ]:
df_games["genres"].isna().mean()

In [ ]:
df_games["name"].duplicated().sum()

In [ ]:
df_games["id"].duplicated().sum()

In [ ]:
df_games["first_release_date"].min(), df_games["first_release_date"].max()

In [ ]:
game_type_names = {
    0: "main_game",             # Juego principal
    1: "dlc_addon",             # DLC o contenido adicional
    2: "expansion",             # Expansión
    3: "bundle",                # Paquete de juegos
    4: "standalone_expansion",  # Expansión independiente
    5: "mod",                   # Modificación
    6: "episode",               # Episodio
    7: "season",                # Temporada
    8: "remake",                # Remake
    9: "remaster",              # Remasterización
    10: "expanded_game",        # Juego ampliado
    11: "port",                 # Adaptación a otra plataforma
    12: "fork",                 # Derivado de otro juego
    13: "pack",                 # Pack
    14: "update"                # Actualización
}

In [ ]:
df_games["game_type_name"] = df_games["game_type"].map(game_type_names)

df_games["game_type_name"].value_counts(dropna=False)

In [ ]:
igdb_endpoints = {
    "genres": "genres",
    "platforms": "platforms",
    "game_modes": "game_modes",
    "player_perspectives": "player_perspectives"
}

In [ ]:
def get_igdb_reference(endpoint, fields="id, name"):

    access_token = get_access_token()

    query = f"""
    fields {fields};
    limit 500;
    """

    headers = {
    "Client-ID": CLIENT_ID,
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json"
    }

    response = requests.post(
        f"{IGDB_BASE_URL}/{endpoint}",
        headers=headers,
        data=query
    )

    response.raise_for_status()

    return pd.DataFrame(response.json())

In [ ]:
df_genres = get_igdb_reference("genres")
df_platforms = get_igdb_reference("platforms")
df_game_modes = get_igdb_reference("game_modes")
df_player_perspectives = get_igdb_reference("player_perspectives")

In [ ]:
display(df_genres)
display(df_platforms)
display(df_game_modes)
display(df_player_perspectives)

In [ ]:
genres_dict = dict(zip(df_genres["id"], df_genres["name"]))

platforms_dict = dict(zip(df_platforms["id"], df_platforms["name"]))

game_modes_dict = dict(zip(df_game_modes["id"], df_game_modes["name"]))

player_perspectives_dict = dict(
    zip(df_player_perspectives["id"], df_player_perspectives["name"])
)

# ["game_modes", "genres", "platforms", "player_perspectives"]

In [ ]:
# para responder a la pregunta qué características se asocian al éxito de un videojuego, filtramos primero por main_game

df_main_games = df_games[
    df_games["game_type"] == 0
].copy()

In [ ]:
all_games = list(all_games)  # Conservamos los registros que ya tenemos
existing_ids = {game["id"] for game in all_games}

for offset in range(5000, 1000000, 500):

    batch = get_games(limit=500, offset=offset)

    if not batch:
        print(f"Fin de los resultados. Offset: {offset}")
        break

    new_games = [
        game for game in batch
        if game["id"] not in existing_ids
    ]

    all_games.extend(new_games)
    existing_ids.update(game["id"] for game in new_games)

    print(
        f"Offset {offset}: recibidos {len(batch)}, "
        f"nuevos {len(new_games)}, "
        f"total acumulado {len(all_games)}"
    )

    # Guardado incremental para no perder lo descargado
    pd.DataFrame(all_games).to_json(
        "igdb_games_raw.json",
        orient="records",
        force_ascii=False
    )

    time.sleep(0.5)

In [ ]:
existing_ids = {game["id"] for game in all_games}

# ajustar start_offset al offset en el que se bloqueó la descarga
start_offset = 0
max_retries = 5

for offset in range(start_offset, 1_000_000, 500):

    # Reintentar la petición si falla temporalmente
    for attempt in range(max_retries):
        try:
            batch = get_games(limit=500, offset=offset)
            break

        except requests.exceptions.HTTPError as e:
            status = e.response.status_code

            # Solo reintentar errores potencialmente temporales
            if status not in (429, 500, 502, 503, 504):
                raise

            if attempt == max_retries - 1:
                # Guardar el progreso antes de detenerse
                pd.DataFrame(all_games).to_json(
                    "igdb_games_checkpoint.json",
                    orient="records",
                    force_ascii=False
                )

                print(f"Error persistente en offset {offset}: {e}")
                print("Progreso guardado. Puedes reanudar desde este offset.")
                raise

            wait_seconds = 5 * (2 ** attempt)
            print(
                f"Error HTTP {status} en offset {offset}. "
                f"Reintento {attempt + 1}/{max_retries} "
                f"en {wait_seconds} segundos..."
            )
            time.sleep(wait_seconds)

    # Si la petición funciona y no devuelve resultados, hemos llegado al final
    if not batch:
        print(f"Fin de los resultados en offset {offset}.")
        break

    # Añadir solo los juegos cuyos IDs aún no tenemos
    new_games = [
        game for game in batch
        if game["id"] not in existing_ids
    ]

    all_games.extend(new_games)
    existing_ids.update(game["id"] for game in new_games)

    print(
        f"Offset {offset}: recibidos {len(batch)}, "
        f"nuevos {len(new_games)}, "
        f"total acumulado {len(all_games)}"
    )

    # Guardar después de cada lote completado
    pd.DataFrame(all_games).to_json(
        "igdb_games_checkpoint.json",
        orient="records",
        force_ascii=False
    )

    time.sleep(0.5)

In [ ]:
pd.DataFrame(all_games).to_json(
    "igdb_games_raw.json",
    orient="records",
    force_ascii=False
)

print(f"Juegos guardados: {len(all_games)}")

# IGDB - STEAM

In [ ]:
# para extraer y guardar la relación IGDB–Steam

def igdb_query(endpoint, query):
    access_token = get_access_token()

    headers = {
        "Client-ID": CLIENT_ID,
        "Authorization": f"Bearer {access_token}"
    }

    response = requests.post(
        f"{BASE_URL}/{endpoint}",
        headers=headers,
        data=query,
        timeout=60
    )

    response.raise_for_status()
    return response.json()

In [ ]:
df_external_sources = pd.DataFrame(
    igdb_query(
        "external_game_sources",
        "fields id, name; limit 500;"
    )
)

display(
    df_external_sources[
        df_external_sources["name"].str.contains(
            "steam", case=False, na=False
        )
    ]
)

In [ ]:
df_external_sources

In [ ]:
# IDs únicos de los juegos que ya descargamos de IGDB

game_ids = (
    df_games["id"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

steam_records = []

# Consultar external_games en lotes de 100 juegos
for start in range(0, len(game_ids), 100):
    batch_ids = game_ids[start:start + 100]
    ids_query = ",".join(map(str, batch_ids))

    query = f"""
        fields id, game, external_game_source, uid, name, url, platform, year;
        where external_game_source = 1 & game = ({ids_query});
        limit 500;
    """

    batch = igdb_query("external_games", query)
    steam_records.extend(batch)

    print(
        f"Procesados {min(start + 100, len(game_ids))}"
        f"/{len(game_ids)} juegos | "
        f"Correspondencias Steam acumuladas: {len(steam_records)}"
    )

    # Pausa para mantenernos por debajo del límite de peticiones
    time.sleep(0.5)

# Convertir el resultado en DataFrame
df_steam_map = pd.DataFrame(steam_records)

print("Correspondencias encontradas:", len(df_steam_map))
display(df_steam_map.head())

# ["uid", "external_game_source"]

uid --> appid

In [ ]:
pd.DataFrame(df_steam_map).to_json(
    "steam_map.json",
    orient="records",
    force_ascii=False
)

In [ ]:
df_games[df_games["id"] == 298060]

In [ ]:
df_steam_map = pd.read_json("steam_map.json")
df_steam_map.head()

In [ ]:
# Comprobamos la cobertura del mapeo
# con esto buscamos responder a tres preguntas: cuántos juegos tienen Steam, cuántos no y si hay juegos con más de una correspondencia.

if not df_steam_map.empty:

    # Evitar repetir exactamente la misma correspondencia
    df_steam_map = df_steam_map.drop_duplicates(
        subset=["game", "uid"]
    ).copy()

    # Número de juegos IGDB con al menos una correspondencia Steam
    matched_games = df_steam_map["game"].nunique()

    # Número de juegos IGDB de nuestra extracción
    total_games = df_games["id"].nunique()

    print(f"Juegos IGDB: {total_games:,}")
    print(f"Juegos con Steam: {matched_games:,}")
    print(f"Juegos sin correspondencia Steam: {total_games - matched_games:,}")
    print(f"Cobertura Steam: {matched_games / total_games:.1%}")

    print(
        "Juegos con más de una correspondencia:",
        df_steam_map["game"].value_counts().gt(1).sum()
    )

    # Guardar la tabla de correspondencias
    df_steam_map.to_json(
        "igdb_steam_mapping.json",
        orient="records",
        force_ascii=False
    )

else:
    print("No se han encontrado correspondencias con Steam.")


""" La cifra de juegos sin correspondencia es orientativa:
puede incluir títulos que no están en Steam,
juegos sin un identificador registrado en IGDB o títulos distribuidos en otras plataformas.
No hacer todavía un merge definitivo con df_games.
Primero comprobaremos si hay varias correspondencias por juego para evitar duplicar filas accidentalmente."""

In [ ]:
# Configuración de conexión y reintentos
session = requests.Session()

retry_strategy = Retry(
    total=6,
    connect=6,
    read=4,
    status=5,
    backoff_factor=3,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

In [ ]:
# Para descargar reseñas de Steam
# purchase_type: 0 = reseñas de usuarios que compraron el juego en Steam;
# 1 = todas las reseñas; 2 = solo reseñas de usuarios que no compraron el juego en Steam

def get_steam_reviews(appid, cursor="*", language="all"):
    url = (
        "https://api.steampowered.com/"
        "IUserReviewsService/GetAppReviews/v1/"
    )

    params = {
        "input_json": json.dumps({
            "appid": int(appid),
            "filter": 1,              # Recent
            "languages": [language],
            "review_type": 0,         # Positivas y negativas
            "purchase_type": 1,       # Todas las reseñas
            "num_per_page": 100,
            "cursor": cursor
        })
    }

    response = session.get(
        url,
        params=params,
        timeout=(15, 90)
    )
    response.raise_for_status()

    result = response.json().get("response", {})

    return {
        "reviews": result.get("reviews", []),
        "query_summary": result.get("query_summary", {}),
        "cursor": result.get("cursor"),
        "total_matching": result.get("total_matching")
    }

In [ ]:
df_steam_map["appid"] = pd.to_numeric(
    df_steam_map["uid"],
    errors="coerce"
)

appids = (
    df_steam_map["appid"]
    .dropna()
    .astype("int64")
    .drop_duplicates()
    #.head(5)
    .tolist()
)

print(len(appids))

In [ ]:
for appid in appids:
    try:
        data = get_steam_reviews(appid)

        reviews = data.get("reviews", [])
        summary = data.get("query_summary", {})

        print(
            f"AppID {appid}: "
            f"reseñas recibidas={len(reviews)}, "
            f"total informado={summary.get('total_reviews', 'N/D')}"
        )

    except requests.exceptions.RequestException as e:
        print(f"Error con AppID {appid}: {e}")

    time.sleep(2)

In [ ]:
def get_all_steam_reviews(appid, max_pages=None):
    all_reviews = []
    seen_ids = set()
    cursor = "*"
    previous_cursors = set()
    page = 0
    summary = {}

    while True:
        result = get_steam_reviews(appid, cursor=cursor)
        page += 1

        # El resumen global suele aparecer en la primera página
        if page == 1:
            summary = result["query_summary"]

        reviews = result["reviews"]

        if not reviews:
            break

        for review in reviews:
            review_id = str(review["recommendationid"])

            if review_id not in seen_ids:
                review["appid"] = int(appid)
                all_reviews.append(review)
                seen_ids.add(review_id)

        print(
            f"AppID {appid} | Página {page} | "
            f"Recibidas {len(reviews)} | "
            f"Acumuladas {len(all_reviews)}"
        )

        next_cursor = result["cursor"]

        if not next_cursor or next_cursor == cursor:
            break

        if next_cursor in previous_cursors:
            print("Cursor repetido: detenemos la paginación.")
            break

        previous_cursors.add(next_cursor)

        if max_pages is not None and page >= max_pages:
            break

        cursor = next_cursor
        time.sleep(1)

    return all_reviews, summary

In [ ]:
reviews_3124580, summary_3124580 = get_all_steam_reviews(
    3124580
)

df_reviews_test = pd.DataFrame(reviews_3124580)

print("Reseñas descargadas:", len(df_reviews_test))
print("Total informado por Steam:", summary_3124580.get("total_reviews"))

display(df_reviews_test.head())

Las reseñas individuales deberían conservar campos como:
- recommendationid: identificador único de la reseña.
- appid: juego al que pertenece.
- review: texto.
- voted_up: recomendación positiva o negativa.
- timestamp_created: fecha de creación.
- language: idioma.
- votes_up: votos útiles.
- author: información de actividad y tiempo de juego disponible.

In [ ]:
# 1. Preparar los mapeos IGDB-Steam
mapping = df_steam_map.copy()

mapping["appid"] = pd.to_numeric(
    mapping["uid"], errors="coerce"
)

mapping = mapping.dropna(subset=["game", "appid"]).copy()
mapping["game"] = mapping["game"].astype(int)
mapping["appid"] = mapping["appid"].astype("int64")

mapping = (
    mapping.drop_duplicates(subset="game")
)

# 2. Unir la información de IGDB con el appid de Steam
candidates = df_games.copy()
candidates["id"] = candidates["id"].astype(int)

candidates = candidates.merge(
    mapping[["game", "appid"]].rename(columns={"game": "id"}),
    on="id",
    how="inner",
    validate="one_to_one"
)

# 3. Preparar variables de estratificación
candidates["release_date"] = pd.to_datetime(
    candidates["first_release_date"],
    unit="s",
    errors="coerce",
    utc=True
)

candidates["release_year"] = candidates["release_date"].dt.year

print("Juegos candidatos:", len(candidates))
print("Años:", candidates["release_year"].min(),
      "-", candidates["release_year"].max())
print("Tipos de juego:", candidates["game_type_name"].nunique())

In [ ]:
game_type_distribution = (
    candidates["game_type_name"]
    .value_counts(dropna=False)
    .rename_axis("game_type")
    .reset_index(name="games")
)

game_type_distribution["percentage"] = (
    game_type_distribution["games"] /
    game_type_distribution["games"].sum() * 100
)

display(game_type_distribution)

In [ ]:
candidates_main = candidates.loc[
    candidates["game_type_name"].eq("main_game")
].copy()

candidates_main = (
    candidates_main
    .drop_duplicates(subset="appid")
    .reset_index(drop=True)
)

print("Juegos candidatos:", len(candidates))
print("Main games:", len(candidates_main))
print("AppID únicos:", candidates_main["appid"].nunique())

In [ ]:
REVIEWS_DIR = Path("steam_etl") / "reviews"
REVIEWS_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = Path("steam_etl") / "manifest.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def extract_one_game(appid):
    start = time.perf_counter()
    extracted_at = datetime.now(timezone.utc).isoformat()

    reviews, summary = get_all_steam_reviews(int(appid))

    # Guardado temporal para reducir el riesgo de dejar un JSON incompleto
    output_path = REVIEWS_DIR / f"appid_{int(appid)}.json"
    temp_path = REVIEWS_DIR / f"appid_{int(appid)}.tmp"

    payload = {
        "appid": int(appid),
        "extracted_at": extracted_at,
        "query_summary": summary,
        "review_count_downloaded": len(reviews),
        "reviews": reviews
    }

    temp_path.write_text(
        json.dumps(payload, ensure_ascii=False),
        encoding="utf-8"
    )
    temp_path.replace(output_path)

    elapsed = time.perf_counter() - start

    return {
        "appid": int(appid),
        "status": "success",
        "reviews_downloaded": len(reviews),
        "reviews_reported": summary.get("total_reviews"),
        "elapsed_seconds": round(elapsed, 2),
        "extracted_at": extracted_at,
        "error": None
    }

In [ ]:
if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
else:
    manifest = pd.DataFrame(columns=[
        "appid",
        "status",
        "reviews_downloaded",
        "reviews_reported",
        "elapsed_seconds",
        "extracted_at",
        "error"
    ])

processed = set(
    manifest.loc[
        manifest["status"].eq("success"), "appid"
    ].dropna().astype(int)
)

#for i, appid in enumerate(df_steam_sample["appid"].astype(int), start=1):
for i, appid in enumerate(candidates_main["appid"].astype(int), start=1):

    output_path = REVIEWS_DIR / f"appid_{appid}.json"

    # Un JSON ya guardado permite reanudar sin repetir una extracción completa
    if appid in processed and output_path.exists():
        continue

    #print(f"[{i}/{len(df_steam_sample)}] Descargando AppID {appid}")
    print(f"[{i}/{len(candidates_main)}] Descargando AppID {appid}")

    try:
        result = extract_one_game(appid)

    except Exception as e:
        result = {
            "appid": appid,
            "status": "failed",
            "reviews_downloaded": None,
            "reviews_reported": None,
            "elapsed_seconds": None,
            "extracted_at": datetime.now(timezone.utc).isoformat(),
            "error": str(e)[:500]
        }

    # Sustituir el registro anterior del mismo AppID, si existe
    manifest = manifest[
        manifest["appid"].astype(str) != str(appid)
    ]

    manifest = pd.concat(
        [manifest, pd.DataFrame([result])],
        ignore_index=True
    )

    manifest.to_csv(MANIFEST_PATH, index=False)

    print(
        f"Estado: {result['status']} | "
        f"Reseñas: {result['reviews_downloaded']} | "
        f"Tiempo: {result['elapsed_seconds']} s"
    )

    time.sleep(1.5)

#######################

In [ ]:
from pathlib import Path
import pandas as pd

PENDING_CSV_PATH = (
    Path("sampling") / "steam_reviews_pending_1689.csv"
)

candidates_main = pd.read_csv(
    PENDING_CSV_PATH,
    dtype={"appid": "int64"}
)

print("Juegos cargados:", len(candidates_main))
print("AppIDs únicos:", candidates_main["appid"].nunique())

#######################

In [ ]:
# preparar siguiente tanda

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
else:
    manifest = pd.DataFrame(columns=[
        "appid",
        "status",
        "reviews_downloaded",
        "reviews_reported",
        "elapsed_seconds",
        "extracted_at",
        "error"
    ])

manifest["appid"] = pd.to_numeric(
    manifest["appid"],
    errors="coerce"
)

# 1. Identificar extracciones completadas
completed = set()

for _, row in manifest.iterrows():

    if row["status"] != "success":
        continue

    if pd.isna(row["appid"]):
        continue

    appid = int(row["appid"])

    output_path = REVIEWS_DIR / f"appid_{appid}.json"

    if output_path.exists():
        completed.add(appid)


# 2. Identificar candidatos pendientes
pending = candidates_main.loc[
    ~candidates_main["appid"].astype(int).isin(completed)
].copy()

pending = pending.drop_duplicates(
    subset="appid"
).reset_index(drop=True)


# 3. Mostrar resumen
print("Main games:", len(candidates_main))
print("Ya descargados:", len(completed))
print("Pendientes:", len(pending))

In [ ]:
BATCH_SIZE = len(pending)
"""BATCH_SIZE = 100"""

batch = pending.head(BATCH_SIZE).copy()

print("Juegos en esta tanda:", len(batch))

In [ ]:
for i, appid in enumerate(
    batch["appid"].astype(int),
    start=1
):

    print(
        f"\n[{i}/{len(batch)}] "
        f"Descargando AppID {appid}"
    )

    try:
        result = extract_one_game(appid)

    except Exception as e:
        result = {
            "appid": appid,
            "status": "failed",
            "reviews_downloaded": None,
            "reviews_reported": None,
            "elapsed_seconds": None,
            "extracted_at": datetime.now(
                timezone.utc
            ).isoformat(),
            "error": str(e)[:500]
        }

    # Sustituir registro anterior
    manifest = manifest.loc[
        manifest["appid"] != appid
    ].copy()

    manifest.loc[len(manifest)] = result

    # Guardar manifiesto
    temp_manifest = MANIFEST_PATH.with_suffix(".tmp")

    manifest.to_csv(temp_manifest, index=False)
    temp_manifest.replace(MANIFEST_PATH)

    print(
        f"Estado: {result['status']} | "
        f"Reseñas: {result['reviews_downloaded']} | "
        f"Tiempo: {result['elapsed_seconds']} s"
    )

    time.sleep(1.5)

print("\nTanda finalizada.")

In [ ]:
# 1. Contar archivos de la carpeta
all_files = list(REVIEWS_DIR.iterdir())

json_files = list(REVIEWS_DIR.glob("appid_*.json"))

print("Elementos totales en carpeta:", len(all_files))
print("Archivos JSON:", len(json_files))
print("Otros elementos:", len(all_files) - len(json_files))


# 2. Extraer los AppID de los nombres de los JSON
json_appids = set()

for path in json_files:
    try:
        appid = int(path.stem.replace("appid_", ""))
        json_appids.add(appid)
    except ValueError:
        print("Nombre de archivo inesperado:", path.name)


# 3. Analizar el manifiesto
manifest_disk = pd.read_csv(MANIFEST_PATH)

manifest_disk["appid"] = pd.to_numeric(
    manifest_disk["appid"],
    errors="coerce"
)

success_manifest = manifest_disk.loc[
    manifest_disk["status"].eq("success")
].dropna(subset=["appid"])

manifest_appids = set(
    success_manifest["appid"].astype(int)
)


# 4. Comparar ambas fuentes
print("\n--- MANIFEST ---")
print("Filas totales:", len(manifest_disk))
print("Filas success:", len(success_manifest))
print("AppID únicos en success:", len(manifest_appids))

print("\n--- JSON ---")
print("Archivos JSON:", len(json_files))
print("AppID únicos en JSON:", len(json_appids))

print("\n--- COMPARACIÓN ---")
print("JSON sin registro success:",
      len(json_appids - manifest_appids))

print("Success sin archivo JSON:",
      len(manifest_appids - json_appids))

print("JSON y manifest coincidentes:",
      len(json_appids & manifest_appids))

Elementos totales en carpeta: 26106
Archivos JSON: 26106
Otros elementos: 0

--- MANIFEST ---
Filas totales: 26108
Filas success: 26106
AppID únicos en success: 26106

--- JSON ---
Archivos JSON: 26106
AppID únicos en JSON: 26106

--- COMPARACIÓN ---
JSON sin registro success: 0
Success sin archivo JSON: 0
JSON y manifest coincidentes: 26106

In [ ]:
candidate_appids = set(
    candidates_main["appid"].dropna().astype(int)
)

print("\n--- RELACIÓN CON CANDIDATES_MAIN ---")

print("JSON que pertenecen a candidates_main:",
      len(json_appids & candidate_appids))

print("JSON que NO pertenecen a candidates_main:",
      len(json_appids - candidate_appids))

print("Candidatos con JSON existente:",
      len(candidate_appids & json_appids))

In [ ]:
import shutil
from datetime import datetime
import json
import pandas as pd

# 1. Copia de seguridad del manifiesto original
backup_path = MANIFEST_PATH.with_name(
    f"manifest_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
)

shutil.copy2(MANIFEST_PATH, backup_path)

print("Copia de seguridad:", backup_path)


# 2. Cargar el manifiesto original
manifest_repaired = pd.read_csv(MANIFEST_PATH)

manifest_repaired["appid"] = pd.to_numeric(
    manifest_repaired["appid"],
    errors="coerce"
)


# 3. Revisar todos los JSON y recuperar registros
recovered = 0
updated = 0
invalid = []

for path in REVIEWS_DIR.glob("appid_*.json"):

    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        appid = int(payload["appid"])
        reviews = payload["reviews"]

        # Comprobar que el JSON tiene la estructura esperada
        if not isinstance(reviews, list):
            raise ValueError("El campo reviews no es una lista")

        if int(payload["review_count_downloaded"]) != len(reviews):
            raise ValueError("El número de reseñas no coincide")

        new_record = {
            "appid": appid,
            "status": "success",
            "reviews_downloaded": len(reviews),
            "reviews_reported": payload["query_summary"].get("total_reviews"),
            "elapsed_seconds": None,
            "extracted_at": payload["extracted_at"],
            "error": None
        }

        existing_rows = manifest_repaired.index[
            manifest_repaired["appid"].eq(appid)
        ].tolist()

        if existing_rows:

            # Si ya existe un registro success, no lo modificamos
            already_success = manifest_repaired.loc[
                existing_rows, "status"
            ].eq("success").any()

            if already_success:
                continue

            # Si existía como error, lo actualizamos
            idx = existing_rows[-1]

            for column, value in new_record.items():
                manifest_repaired.loc[idx, column] = value

            updated += 1

        else:

            # Si no existía, añadimos el registro recuperado
            manifest_repaired = pd.concat(
                [
                    manifest_repaired,
                    pd.DataFrame([new_record])
                ],
                ignore_index=True
            )

            recovered += 1

    except Exception as e:
        invalid.append({
            "file": path.name,
            "error": str(e)
        })


# 4. Guardar el manifiesto reparado
manifest_repaired.to_csv(
    MANIFEST_PATH,
    index=False
)

print("\n--- RESULTADO DE LA REPARACIÓN ---")
print("Registros recuperados:", recovered)
print("Registros de error actualizados:", updated)
print("Archivos JSON no válidos:", len(invalid))
print("Total de filas en el manifest:", len(manifest_repaired))
print("\nEstados:")
print(manifest_repaired["status"].value_counts(dropna=False))

if invalid:
    print("\nArchivos que requieren revisión:")
    display(pd.DataFrame(invalid))

In [ ]:
failed = manifest.loc[
    manifest["status"].eq("failed")
].copy()

print("Extracciones fallidas:", len(failed))

display(
    failed["error"]
    .value_counts(dropna=False)
    .head(20)
)

Juegos con archivo: 10363

Total de reseñas: 5594049

"total_positive": 912, "total_negative": 224,

In [ ]:
game_records = []

json_files = list(REVIEWS_DIR.glob("appid_*.json"))

for i, path in enumerate(json_files, start=1):

    with path.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    game_records.append({
        "appid": payload["appid"],
        "extracted_at": payload["extracted_at"],
        "review_count_downloaded": payload["review_count_downloaded"],
        "reviews_reported": payload["query_summary"].get("total_reviews")#,
        #"positive_reviews": payload["query_summary"].get("total_positive"),
        #"negative_reviews": payload["query_summary"].get("total_negative")
    })

    if i % 1000 == 0:
        print(f"Procesados {i}/{len(json_files)} juegos")

df_steam_manifest = pd.DataFrame(game_records)

print("Juegos con archivo:", df_steam_manifest["appid"].nunique())

print(
    "Total de reseñas descargadas:",
    df_steam_manifest["review_count_downloaded"].sum()
)

In [ ]:
# Cargar el manifiesto operativo
df_steam_manifest = pd.read_csv(MANIFEST_PATH)

# Normalizar tipos
df_steam_manifest["appid"] = pd.to_numeric(
    df_steam_manifest["appid"],
    errors="coerce"
)

df_steam_manifest["reviews_downloaded"] = pd.to_numeric(
    df_steam_manifest["reviews_downloaded"],
    errors="coerce"
)

# Eliminar posibles duplicados de AppID
df_steam_manifest = (
    df_steam_manifest
    .dropna(subset=["appid"])
    .drop_duplicates(subset="appid", keep="last")
)

# Considerar únicamente extracciones correctas
df_steam_manifest = df_steam_manifest.loc[
    df_steam_manifest["status"].eq("success")
].copy()

# Calcular estadísticas
print("Juegos descargados:",
      df_steam_manifest["appid"].nunique())

print("Total de reseñas descargadas:",
      df_steam_manifest["reviews_downloaded"].sum())

print("Media de reseñas por juego:",
      df_steam_manifest["reviews_downloaded"].mean())

print("Mediana de reseñas por juego:",
      df_steam_manifest["reviews_downloaded"].median())

print("Máximo de reseñas de un juego:",
      df_steam_manifest["reviews_downloaded"].max())

La distribución está muy sesgada. Unos pocos juegos concentran una proporción enorme de las reseñas

In [ ]:
print("Archivos JSON:", len(list(REVIEWS_DIR.glob("appid_*.json"))))

print(
    "Total de reseñas descargadas:",
    df_steam_manifest["review_count_downloaded"].sum()
)

print(
    "Tamaño total de los JSON (GB):",
    round(
        sum(
            p.stat().st_size
            for p in REVIEWS_DIR.glob("appid_*.json")
        ) / (1024 ** 3),
        2
    )
)

In [ ]:
df_steam_reviews.to_json(
    "steam_reviews.json",
    orient="records",
    force_ascii=False
)
"""df_steam_reviews.to_parquet(
    "steam_reviews.parquet",
    index=False
)"""

df_steam_manifest.to_csv(
    "steam_reviews_game_summary.csv",
    index=False
)

"""df_steam_manifest.to_parquet(
    "steam_reviews_game_summary.parquet",
    index=False
)"""

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)

print("Estados de extracción:")
display(manifest["status"].value_counts(dropna=False))

print("Resumen de tiempos:")
display(manifest["elapsed_seconds"].describe())

print("Tiempo total registrado (minutos):",
      round(manifest["elapsed_seconds"].sum() / 60, 2))

print("Reseñas descargadas:",
      manifest["reviews_downloaded"].sum())

print("Juegos con error:",
      (manifest["status"] != "success").sum())

In [ ]:
print("Filas de reseñas:", len(df_steam_reviews))

if "recommendationid" in df_steam_reviews.columns:
    print(
        "IDs de reseña duplicados:",
        df_steam_reviews["recommendationid"].duplicated().sum()
    )

quality_cols = [
    c for c in [
        "recommendationid",
        "appid",
        "review",
        "voted_up",
        "timestamp_created",
        "language"
    ]
    if c in df_steam_reviews.columns
]

display(
    df_steam_reviews[quality_cols]
    .isna()
    .sum()
    .to_frame("missing_values")
)

In [ ]:
if "voted_up" in df_steam_reviews.columns:
    display(
        df_steam_reviews["voted_up"]
        .value_counts(dropna=False)
        .rename(index={True: "Positive", False: "Negative"})
        .to_frame("reviews")
    )

    print(
        "Porcentaje positivo:",
        round(df_steam_reviews["voted_up"].mean() * 100, 2),
        "%"
    )

In [ ]:
comparison = df_steam_manifest.copy()

comparison["reviews_reported"] = pd.to_numeric(
    comparison["reviews_reported"], errors="coerce"
)

comparison["difference"] = (
    comparison["reviews_reported"]
    - comparison["review_count_downloaded"]
)

display(
    comparison.sort_values(
        "difference", ascending=False
    ).head(20)
)

print(
    "Juegos cuyo total descargado coincide con el informado:",
    (comparison["difference"] == 0).sum(),
    "de",
    comparison["difference"].notna().sum()
)

In [ ]:
comparison = df_steam_manifest.copy()

comparison["reviews_reported"] = pd.to_numeric(
    comparison["reviews_reported"], errors="coerce"
)
comparison["review_count_downloaded"] = pd.to_numeric(
    comparison["review_count_downloaded"], errors="coerce"
)

comparison["difference"] = (
    comparison["reviews_reported"]
    - comparison["review_count_downloaded"]
)

print("Juegos totales:", len(comparison))
print("Sin total informado:", comparison["reviews_reported"].isna().sum())
print("Con coincidencia:", (comparison["difference"] == 0).sum())
print(
    "Con discrepancia:",
    (
        comparison["difference"].notna()
        & comparison["difference"].ne(0)
    ).sum()
)

display(
    comparison.loc[
        comparison["difference"].notna()
        & comparison["difference"].ne(0),
        [
            "appid",
            "reviews_reported",
            "review_count_downloaded",
            "difference",
        ],
    ].sort_values("difference", ascending=False)
)

Las diferencias pueden tener varias causas:
- Las reseñas pueden haberse publicado o modificado mientras se descargaban las páginas.
- El total informado por Steam puede variar con los filtros y el momento de la consulta.
- Puede haber un problema de paginación o una interrupción que no se haya registrado como error.
Si las discrepancias son pequeñas, pueden deberse a cambios durante la extracción. Si son grandes, investigaremos esos juegos individualmente. No marcaría automáticamente las 16 extracciones como fallidas.

In [ ]:
# Confirmar que no estamos midiendo solo reseñas en inglés

display(
    df_steam_reviews["language"]
    .value_counts(dropna=False)
    .head(20)
    .to_frame("reviews")
)

In [ ]:
# Porcentaje de textos vacíos

review_text = df_steam_reviews["review"].fillna("").astype(str).str.strip()

print("Textos vacíos:", (review_text == "").sum())
print("Porcentaje de textos vacíos:", round(review_text.eq("").mean() * 100, 3), "%")

# ITAD

In [ ]:
load_dotenv()

ITAD_API_KEY = os.getenv("ITAD_API_KEY")

if not ITAD_API_KEY:
    raise ValueError("No se ha encontrado ITAD_API_KEY en el archivo .env")

ITAD_BASE_URL = "https://api.isthereanydeal.com"

print("API key cargada correctamente.")

In [ ]:
df_sample = candidates_main.copy()

df_sample["appid"] = pd.to_numeric(
    df_sample["appid"], errors="coerce"
)

df_sample = (
    df_sample
    .dropna(subset=["appid"])
    .drop_duplicates(subset=["appid"])
    .copy()
)

df_sample["appid"] = df_sample["appid"].astype("int64")

print("Juegos únicos con appid:", len(df_sample))
display(df_sample[["appid"]].head())

In [ ]:
# Función para crear correspondencias entre Steam e ITAD
def get_itad_game_ids(appids, batch_size=200):
    url = f"{ITAD_BASE_URL}/lookup/id/shop/61/v1"

    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    records = []

    for start in range(0, len(appids), batch_size):
        batch = appids[start:start + batch_size]

        shop_ids = [f"app/{appid}" for appid in batch]

        response = requests.post(
            url,
            params={"key": ITAD_API_KEY},
            headers=headers,
            json=shop_ids,
            timeout=60,
        )
        response.raise_for_status()

        result = response.json()

        for appid in batch:
            shop_id = f"app/{appid}"
            records.append({
                "appid": appid,
                "itad_id": result.get(shop_id),
            })

        print(
            f"Procesados {min(start + len(batch), len(appids))}"
            f" de {len(appids)} juegos"
        )

        time.sleep(0.5)

    return pd.DataFrame(records)

In [ ]:
# Tabla de correspondencias entre Steam e ITAD

appids = df_sample["appid"].tolist()

df_itad_mapping = get_itad_game_ids(appids)

print("Total de juegos:", len(df_itad_mapping))
print("Con ID de ITAD:", df_itad_mapping["itad_id"].notna().sum())
print("Sin ID de ITAD:", df_itad_mapping["itad_id"].isna().sum())

display(df_itad_mapping.head())

In [ ]:
def get_retry_wait(response, attempt):

    retry_after = response.headers.get("Retry-After")

    if retry_after:
        try:
            # Retry-After expresado en segundos
            return max(1, float(retry_after))

        except ValueError:
            try:
                # Retry-After expresado como fecha HTTP
                retry_date = parsedate_to_datetime(retry_after)

                if retry_date.tzinfo is None:
                    retry_date = retry_date.replace(tzinfo=timezone.utc)

                wait = (
                    retry_date - datetime.now(timezone.utc)
                ).total_seconds()

                return max(1, wait)

            except (TypeError, ValueError):
                pass

    # Espera exponencial si no existe una cabecera válida
    return 60 * (2 ** attempt)

In [ ]:
def post_itad_with_retry(
    url,
    headers,
    params,
    json_data,
    max_retries=6
):

    for attempt in range(max_retries):

        response = requests.post(
            url,
            params=params,
            headers=headers,
            json=json_data,
            timeout=60
        )

        # Petición correcta
        if response.status_code == 200:
            return response

        # Rate limit
        if response.status_code == 429:

            wait = get_retry_wait(response, attempt)

            print(
                f"429 Too Many Requests. "
                f"Intento {attempt + 1}/{max_retries}. "
                f"Esperando {wait:.1f} segundos."
            )

            time.sleep(wait)
            continue

        # Errores temporales del servidor
        if response.status_code in [500, 502, 503, 504]:

            wait = 10 * (2 ** attempt)

            print(
                f"Error {response.status_code}. "
                f"Esperando {wait} segundos."
            )

            time.sleep(wait)
            continue

        # Otros errores: no reintentar automáticamente
        response.raise_for_status()

    raise RuntimeError(
        f"ITAD no respondió correctamente después de {max_retries} intentos."
    )

In [ ]:
def get_itad_game_ids(
    appids,
    batch_size=200,
    delay=1,
    checkpoint_path="itad_mapping_checkpoint.csv"
):

    url = f"{ITAD_BASE_URL}/lookup/id/shop/61/v1"

    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    # Limpiar y normalizar AppIDs
    appids = (
        pd.Series(appids)
        .dropna()
        .astype("int64")
        .drop_duplicates()
        .tolist()
    )

    # Cargar progreso anterior
    if Path(checkpoint_path).exists():

        df_checkpoint = pd.read_csv(checkpoint_path)

        processed_appids = set(
            pd.to_numeric(
                df_checkpoint["appid"],
                errors="coerce"
            )
            .dropna()
            .astype("int64")
        )

        records = df_checkpoint.to_dict("records")

    else:

        df_checkpoint = pd.DataFrame(
            columns=["appid", "itad_id"]
        )

        processed_appids = set()
        records = []

    # AppIDs pendientes
    pending_appids = [
        appid for appid in appids
        if appid not in processed_appids
    ]

    print("Total de AppIDs:", len(appids))
    print("Ya procesados:", len(processed_appids))
    print("Pendientes:", len(pending_appids))

    # Procesar lotes
    for start in range(0, len(pending_appids), batch_size):

        batch = pending_appids[start:start + batch_size]

        shop_ids = [
            f"app/{appid}" for appid in batch
        ]

        print(
            f"\nProcesando lote "
            f"{start // batch_size + 1} "
            f"({len(batch)} AppIDs)"
        )

        try:

            response = post_itad_with_retry(
                url=url,
                headers=headers,
                params={"key": ITAD_API_KEY},
                json_data=shop_ids
            )

            result = response.json()

            for appid in batch:

                shop_id = f"app/{appid}"

                records.append({
                    "appid": appid,
                    "itad_id": result.get(shop_id)
                })

            # Actualizar checkpoint
            df_checkpoint = pd.DataFrame(records)

            temp_path = str(checkpoint_path) + ".tmp"

            df_checkpoint.to_csv(
                temp_path,
                index=False
            )

            Path(temp_path).replace(checkpoint_path)

            processed_appids.update(batch)

            print(
                f"Procesados: {len(processed_appids)} "
                f"/ {len(appids)}"
            )

        except Exception as e:

            print(f"Error procesando lote: {e}")
            print("El progreso anterior está guardado.")
            raise

        # Pausa entre peticiones
        time.sleep(delay)

    return pd.DataFrame(records)

In [ ]:
appids = (
    candidates_main["appid"]
    .dropna()
    .astype("int64")
    .drop_duplicates()
    .tolist()
)

df_itad_mapping = get_itad_game_ids(
    appids,
    batch_size=200,
    delay=1,
    checkpoint_path="itad_mapping_checkpoint.csv"
)

In [ ]:
print("Total de juegos:", len(df_itad_mapping))

print(
    "Con ID de ITAD:",
    df_itad_mapping["itad_id"].notna().sum()
)

print(
    "Sin ID de ITAD:",
    df_itad_mapping["itad_id"].isna().sum()
)

display(df_itad_mapping.head())

In [ ]:
df_itad_mapping.to_csv("itad_mapping.csv", index=False)

In [ ]:
if not ITAD_API_KEY:
    raise ValueError("Falta ITAD_API_KEY en el archivo .env")

ITAD_BASE_URL = "https://api.isthereanydeal.com"
COUNTRY = "ES"

OUTPUT_DIR = Path("itad_etl")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_itad_overview(game_ids, batch_size=200):
    url = f"{ITAD_BASE_URL}/games/overview/v2"
    headers = {"Accept": "application/json"}

    results = []
    errors = []

    valid_ids = [
        str(game_id)
        for game_id in game_ids
        if pd.notna(game_id) and str(game_id).strip()
    ]

    for start in range(0, len(valid_ids), batch_size):
        batch = valid_ids[start:start + batch_size]

        try:
            response = requests.post(
                url,
                params={
                    "key": ITAD_API_KEY,
                    "country": COUNTRY,
                },
                headers=headers,
                json=batch,
                timeout=60,
            )
            response.raise_for_status()
            payload = response.json()

            # Guardamos la respuesta original para trazabilidad.
            with (OUTPUT_DIR / f"overview_{start:04d}.json").open(
                "w", encoding="utf-8"
            ) as f:
                json.dump(payload, f, ensure_ascii=False, indent=2)

            for item in payload.get("prices", []):
                current = item.get("current") or {}
                lowest = item.get("lowest") or {}

                current_price = current.get("price") or {}
                current_regular = current.get("regular") or {}
                lowest_price = lowest.get("price") or {}
                lowest_regular = lowest.get("regular") or {}

                results.append({
                    "itad_id": item.get("id"),
                    "current_shop": (current.get("shop") or {}).get("name"),
                    "current_price": current_price.get("amount"),
                    "current_currency": current_price.get("currency"),
                    "current_regular_price": current_regular.get("amount"),
                    "current_discount_pct": current.get("cut"),
                    "current_price_timestamp": current.get("timestamp"),
                    "current_url": current.get("url"),
                    "historical_low_shop": (lowest.get("shop") or {}).get("name"),
                    "historical_low_price": lowest_price.get("amount"),
                    "historical_low_currency": lowest_price.get("currency"),
                    "historical_low_regular_price": lowest_regular.get("amount"),
                    "historical_low_discount_pct": lowest.get("cut"),
                    "historical_low_timestamp": lowest.get("timestamp"),
                    "bundled": item.get("bundled"),
                    "itad_url": (item.get("urls") or {}).get("game"),
                })

            print(
                f"Lote {start // batch_size + 1}: "
                f"{min(start + len(batch), len(valid_ids))}"
                f"/{len(valid_ids)} IDs enviados"
            )

        except requests.RequestException as exc:
            errors.append({
                "batch_start": start,
                "game_ids": batch,
                "error": str(exc),
            })
            print(f"Error en el lote desde {start}: {exc}")

        time.sleep(0.5)

    return pd.DataFrame(results), errors

In [ ]:
# Partimos de la tabla de correspondencias ya creada
df_mapping = df_itad_mapping[["appid", "itad_id"]].copy()

df_mapping["appid"] = pd.to_numeric(
    df_mapping["appid"], errors="coerce"
)

df_mapping = df_mapping.dropna(subset=["appid"]).copy()
df_mapping["appid"] = df_mapping["appid"].astype("int64")

# Solo consultamos IDs de ITAD válidos.
itad_ids = (
    df_mapping["itad_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

df_itad_prices, itad_errors = get_itad_overview(itad_ids)

df_itad_prices.to_csv(
    OUTPUT_DIR / "itad_overview.csv",
    index=False,
)

with (OUTPUT_DIR / "itad_errors.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(itad_errors, f, ensure_ascii=False, indent=2)

print("IDs ITAD enviados:", len(itad_ids))
print("Juegos devueltos por ITAD:", len(df_itad_prices))
print("Lotes con errores:", len(itad_errors))

display(df_itad_prices.head())

In [ ]:
OVERVIEW_MANIFEST_PATH = OUTPUT_DIR / "overview_manifest.csv"
OVERVIEW_ERRORS_PATH = OUTPUT_DIR / "itad_errors.json"
OVERVIEW_FINAL_PATH = OUTPUT_DIR / "itad_overview.csv"

In [ ]:
def get_retry_wait(response, attempt):

    retry_after = response.headers.get("Retry-After")

    if retry_after:

        try:
            return max(1, float(retry_after))

        except ValueError:

            try:
                retry_date = parsedate_to_datetime(retry_after)

                if retry_date.tzinfo is None:
                    retry_date = retry_date.replace(
                        tzinfo=timezone.utc
                    )

                wait = (
                    retry_date - datetime.now(timezone.utc)
                ).total_seconds()

                return max(1, wait)

            except (TypeError, ValueError):
                pass

    return 60 * (2 ** attempt)

In [ ]:
def post_itad_with_retry(
    url,
    headers,
    params,
    json_data,
    max_retries=6
):

    for attempt in range(max_retries + 1):

        try:

            response = requests.post(
                url,
                params=params,
                headers=headers,
                json=json_data,
                timeout=60
            )

        except (requests.Timeout, requests.ConnectionError):

            if attempt == max_retries:
                raise

            wait = 30 * (2 ** attempt)

            print(
                f"Error de conexión. "
                f"Esperando {wait} segundos."
            )

            time.sleep(wait)
            continue

        if response.status_code == 200:
            return response

        if response.status_code == 429 or response.status_code in [
            500, 502, 503, 504
        ]:

            if attempt == max_retries:
                response.raise_for_status()

            if response.status_code == 429:
                wait = get_retry_wait(response, attempt)
            else:
                wait = 30 * (2 ** attempt)

            print(
                f"HTTP {response.status_code}. "
                f"Intento {attempt + 1}/{max_retries}. "
                f"Esperando {wait:.1f} segundos."
            )

            time.sleep(wait + 1)
            continue

        response.raise_for_status()

    raise RuntimeError("No se pudo completar la petición a ITAD.")

In [ ]:
OVERVIEW_COLUMNS = [
    "itad_id",
    "current_shop",
    "current_price",
    "current_currency",
    "current_regular_price",
    "current_discount_pct",
    "current_price_timestamp",
    "current_url",
    "historical_low_shop",
    "historical_low_price",
    "historical_low_currency",
    "historical_low_regular_price",
    "historical_low_discount_pct",
    "historical_low_timestamp",
    "bundled",
    "itad_url"
]


def normalize_itad_overview(payload):

    results = []

    for item in payload.get("prices", []):

        current = item.get("current") or {}
        lowest = item.get("lowest") or {}

        current_price = current.get("price") or {}
        current_regular = current.get("regular") or {}

        lowest_price = lowest.get("price") or {}
        lowest_regular = lowest.get("regular") or {}

        results.append({

            "itad_id": item.get("id"),

            "current_shop": (
                current.get("shop") or {}
            ).get("name"),

            p8: item.get("bundled"),

            "itad_url": (
                item.get("urls") or {}
            ).get("game")
        })

    return pd.DataFrame(
        results,
        columns=OVERVIEW_COLUMNS
    )

In [ ]:
def get_itad_overview(
    game_ids,
    batch_size=200,
    delay=2
):

    url = f"{ITAD_BASE_URL}/games/overview/v2"

    headers = {
        "Accept": "application/json"
    }

    # Normalizar IDs y eliminar duplicados
    valid_ids = (
        pd.Series(game_ids)
        .dropna()
        .astype(str)
        .str.strip()
    )

    valid_ids = valid_ids[
        valid_ids.ne("")
    ].drop_duplicates().tolist()

    total_batches = (
        len(valid_ids) + batch_size - 1
    ) // batch_size

    errors = []

    print("Total de IDs:", len(valid_ids))
    print("Total de lotes:", total_batches)

    for batch_number, start in enumerate(
        range(0, len(valid_ids), batch_size),
        start=1
    ):

        batch = valid_ids[start:start + batch_size]

        # Archivos de cada lote
        json_path = OUTPUT_DIR / f"overview_{start:04d}.json"

        csv_path = OUTPUT_DIR / f"overview_prices_{start:06d}.csv"

        print(
            f"\nLote {batch_number}/{total_batches} "
            f"({len(batch)} IDs)"
        )

        # 1. Si el CSV ya existe, omitir
        if csv_path.exists():

            print("  -> CSV existente. Se omite.")
            continue

        # 2. Si existe JSON, reutilizarlo
        if json_path.exists():

            print("  -> Reutilizando JSON existente.")

            try:

                with json_path.open("r", encoding="utf-8") as f:
                    payload = json.load(f)

            except (json.JSONDecodeError, OSError) as e:

                print(f"  -> JSON no válido: {e}")

                errors.append({
                    "batch_start": start,
                    "error": str(e)
                })

                continue

        else:

            # 3. Consultar ITAD
            try:

                response = post_itad_with_retry(
                    url=url,
                    headers=headers,
                    params={
                        "key": ITAD_API_KEY,
                        "country": COUNTRY
                    },
                    json_data=batch
                )

                payload = response.json()

                # Guardado atómico del JSON original
                temp_json = json_path.with_suffix(".json.tmp")

                with temp_json.open("w", encoding="utf-8") as f:
                    json.dump(
                        payload,
                        f,
                        ensure_ascii=False,
                        indent=2
                    )

                temp_json.replace(json_path)

                print("  -> Respuesta descargada.")

            except Exception as e:

                print(f"  -> Error: {e}")

                errors.append({
                    "batch_start": start,
                    "error": str(e)
                })

                # Guardar errores y detener la extracción.
                # No seguimos enviando peticiones si ITAD
                # sigue rechazando nuestras solicitudes.
                with OVERVIEW_ERRORS_PATH.open(
                    "w", encoding="utf-8"
                ) as f:
                    json.dump(
                        errors,
                        f,
                        ensure_ascii=False,
                        indent=2
                    )

                break

            # Pausa entre peticiones nuevas
            time.sleep(delay)

        # 4. Transformar respuesta
        try:

            df_batch = normalize_itad_overview(payload)

            # Guardado atómico del CSV del lote
            temp_csv = csv_path.with_suffix(".csv.tmp")

            df_batch.to_csv(
                temp_csv,
                index=False
            )

            temp_csv.replace(csv_path)

            print(
                f"  -> Guardadas {len(df_batch)} "
                "filas en CSV."
            )

        except Exception as e:

            print(f"  -> Error transformando el lote: {e}")

            errors.append({
                "batch_start": start,
                "error": str(e)
            })

            break

    # 5. Consolidar todos los CSV de los lotes esperados
    batch_csv_paths = [
        OUTPUT_DIR / f"overview_prices_{start:06d}.csv"
        for start in range(0, len(valid_ids), batch_size)
    ]

    existing_csvs = [
        path for path in batch_csv_paths
        if path.exists()
    ]

    if existing_csvs:

        df_itad_prices = pd.concat(
            [
                pd.read_csv(path)
                for path in existing_csvs
            ],
            ignore_index=True
        )

    else:

        df_itad_prices = pd.DataFrame(
            columns=OVERVIEW_COLUMNS
        )

    # Guardar resultado consolidado
    temp_final = OVERVIEW_FINAL_PATH.with_suffix(".csv.tmp")

    df_itad_prices.to_csv(
        temp_final,
        index=False
    )

    temp_final.replace(OVERVIEW_FINAL_PATH)

    # Guardar errores
    with OVERVIEW_ERRORS_PATH.open(
        "w", encoding="utf-8"
    ) as f:

        json.dump(
            errors,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\n" + "=" * 45)
    print("RESUMEN DE EXTRACCIÓN")
    print("=" * 45)

    print("Lotes con CSV:", len(existing_csvs))
    print("Lotes totales:", total_batches)
    print("IDs enviados:", len(valid_ids))
    print("Registros de precios:", len(df_itad_prices))
    print("Errores en esta ejecución:", len(errors))

    return df_itad_prices, errors

In [ ]:
itad_ids = (
    df_mapping["itad_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("IDs únicos de ITAD:", len(itad_ids))

In [ ]:
df_itad_prices, itad_errors = get_itad_overview(
    itad_ids,
    batch_size=200,
    delay=2
)

In [ ]:
print("Juegos con información de precios:", len(df_itad_prices))

print(
    "IDs ITAD únicos devueltos:",
    df_itad_prices["itad_id"].nunique()
)

print(
    "Juegos con precio actual:",
    df_itad_prices["current_price"].notna().sum()
)

print(
    "Juegos con histórico mínimo:",
    df_itad_prices["historical_low_price"].notna().sum()
)

print(
    "Errores:",
    len(itad_errors)
)

display(df_itad_prices.head())

In [ ]:
df_itad_prices.to_csv("itad_prices.csv", index=False)

In [ ]:
# Una fila por appid en la tabla final de cobertura.
df_itad_coverage = df_mapping.merge(
    df_itad_prices,
    on="itad_id",
    how="left",
    validate="many_to_one",
)

print("Juegos de la muestra:", df_itad_coverage["appid"].nunique())
print(
    "Con ID ITAD:",
    df_itad_coverage["itad_id"].notna().sum()
)
print(
    "Con precio actual:",
    df_itad_coverage["current_price"].notna().sum()
)
print(
    "Con mínimo histórico:",
    df_itad_coverage["historical_low_price"].notna().sum()
)
print(
    "Sin precio actual (%):",
    round(df_itad_coverage["current_price"].isna().mean() * 100, 2)
)

display(
    df_itad_coverage[
        [
            "appid",
            "itad_id",
            "current_price",
            "current_currency",
            "current_shop",
            "historical_low_price",
            "historical_low_currency",
        ]
    ].head(20)
)

df_itad_coverage.to_csv(
    OUTPUT_DIR / "itad_coverage_500.csv",
    index=False,
)

In [ ]:
# Elegir un juego con precio actual y mínimo histórico.
test_candidates = df_itad_coverage.loc[
    df_itad_coverage["current_price"].notna()
    & df_itad_coverage["historical_low_price"].notna()
    & df_itad_coverage["itad_id"].notna()
]

if test_candidates.empty:
    raise ValueError("No hay juegos adecuados para probar el historial.")

test_game = test_candidates.iloc[0]

url = f"{ITAD_BASE_URL}/games/history/v2"

response = requests.get(
    url,
    params={
        "key": ITAD_API_KEY,
        "id": test_game["itad_id"],
        "country": "ES",
        # Probamos inicialmente con los últimos 3 meses.
    },
    timeout=60,
)

print("HTTP status:", response.status_code)
response.raise_for_status()

history_test = response.json()

print("appid:", test_game["appid"])
print("itad_id:", test_game["itad_id"])
print("Tipo de respuesta:", type(history_test).__name__)
print("Registros históricos:", len(history_test))
display(pd.json_normalize(history_test).head(10))

In [ ]:
# Inspeccionar la estructura real de un registro
if history_test:
    print(json.dumps(history_test[0], indent=2, ensure_ascii=False))
else:
    print("La API no ha devuelto registros históricos.")

In [ ]:
if history_test:
    print("Claves del primer registro:", list(history_test[0].keys()))

    df_history_test = pd.json_normalize(history_test)

    print("Dimensiones:", df_history_test.shape)
    display(df_history_test.head(10))

In [ ]:
SINCE = "2010-01-01T00:00:00Z"

HISTORY_DIR = Path("itad_etl") / "history"
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

HISTORY_MANIFEST_PATH = Path("itad_etl") / "history_manifest.csv"
HISTORY_CSV_PATH = Path("itad_etl") / "itad_price_history.csv"

HISTORY_URL = f"{ITAD_BASE_URL}/games/history/v2"

In [ ]:
def fetch_history(itad_id, appid, max_retries=5, initial_wait=10):

    params = {
        "key": ITAD_API_KEY,
        "id": str(itad_id),
        "country": COUNTRY,
        "since": SINCE,
    }

    for attempt in range(max_retries + 1):

        try:
            response = requests.get(
                HISTORY_URL,
                params=params,
                timeout=60,
            )

            # Rate limit.
            if response.status_code == 429:

                if attempt == max_retries:
                    raise ITADRateLimitError(
                        f"HTTP 429 persistente para appid={appid}"
                    )

                wait = get_retry_wait(
                    response, attempt, initial_wait
                )

                print(
                    f"HTTP 429 | appid={appid} | "
                    f"reintento {attempt + 1}/{max_retries} "
                    f"en {wait:.1f} segundos"
                )

                time.sleep(wait)
                continue

            # Errores temporales del servidor.
            if response.status_code in (500, 502, 503, 504, 521, 522, 523, 524):

                if attempt == max_retries:
                    response.raise_for_status()

                wait = get_retry_wait(
                    response, attempt, initial_wait
                )

                print(
                    f"HTTP {response.status_code} | appid={appid} | "
                    f"reintento {attempt + 1}/{max_retries} "
                    f"en {wait:.1f} segundos"
                )

                time.sleep(wait)
                continue

            response.raise_for_status()

            payload = response.json()

            if not isinstance(payload, list):
                raise ValueError(
                    f"Formato inesperado para appid={appid}: "
                    f"{type(payload).__name__}"
                )

            return payload

        except ITADRateLimitError:
            raise

        except requests.RequestException as exc:

            if attempt == max_retries:
                raise

            wait = initial_wait * (2 ** attempt)
            wait += random.uniform(1, 3)

            print(
                f"Error de conexión | appid={appid} | "
                f"reintento {attempt + 1}/{max_retries} "
                f"en {wait:.1f} segundos: {exc}"
            )

            time.sleep(wait)

    raise RuntimeError(
        f"Se agotaron los reintentos para appid={appid}"
    )

In [ ]:
class ITADRateLimitError(Exception):
    """Se han agotado los reintentos por límite de peticiones."""
    pass

In [ ]:
def get_retry_wait(response, attempt, initial_wait=5):
    """Calcula la espera respetando Retry-After."""

    retry_after = response.headers.get("Retry-After")

    if retry_after:
        # Puede venir expresado en segundos.
        try:
            return max(0, float(retry_after)) + random.uniform(1, 3)
        except ValueError:
            pass

        # O puede venir como fecha HTTP.
        try:
            retry_date = parsedate_to_datetime(retry_after)

            if retry_date.tzinfo is None:
                retry_date = retry_date.replace(tzinfo=timezone.utc)

            seconds = (
                retry_date - datetime.now(timezone.utc)
            ).total_seconds()

            return max(0, seconds) + random.uniform(1, 3)

        except (TypeError, ValueError, OverflowError):
            pass

    # Backoff exponencial si no hay Retry-After interpretable.
    return initial_wait * (2 ** attempt) + random.uniform(1, 3)

In [ ]:
def normalize_history(appid, itad_id, history):
    """Convierte el historial anidado en filas tabulares."""
    records = []

    for item in history:
        shop = item.get("shop") or {}
        deal = item.get("deal") or {}
        price = deal.get("price") or {}
        regular = deal.get("regular") or {}

        records.append({
            "appid": int(appid),
            "itad_id": str(itad_id),
            "timestamp": item.get("timestamp"),
            "shop_id": shop.get("id"),
            "shop_name": shop.get("name"),
            "price_amount": price.get("amount"),
            "price_amount_int": price.get("amountInt"),
            "price_currency": price.get("currency"),
            "regular_amount": regular.get("amount"),
            "regular_amount_int": regular.get("amountInt"),
            "regular_currency": regular.get("currency"),
            "discount_pct": deal.get("cut"),
        })

    return records

In [ ]:
# Partimos de la correspondencia existente.
df_mapping = df_itad_mapping[["appid", "itad_id"]].copy()
df_mapping["appid"] = pd.to_numeric(
    df_mapping["appid"], errors="coerce"
)
df_mapping = df_mapping.dropna(subset=["appid"]).copy()
df_mapping["appid"] = df_mapping["appid"].astype("int64")

df_mapping = df_mapping.drop_duplicates(subset=["appid"])

In [ ]:
# Leer el manifiesto previo para permitir reanudar.
if HISTORY_MANIFEST_PATH.exists():
    old_manifest = pd.read_csv(HISTORY_MANIFEST_PATH)
else:
    old_manifest = pd.DataFrame(
        columns=[
            "appid", "itad_id", "status", "records_downloaded",
            "elapsed_seconds", "extracted_at", "error"
        ]
    )

manifest_by_appid = {
    int(row["appid"]): row.to_dict()
    for _, row in old_manifest.iterrows()
    if pd.notna(row.get("appid"))
}

In [ ]:
# 2.0
all_records = []
new_manifest_rows = []

stop_extraction = False

for row in df_mapping.itertuples(index=False):
    appid = int(row.appid)

    if pd.isna(row.itad_id) or not str(row.itad_id).strip():
        new_manifest_rows.append({
            "appid": appid,
            "itad_id": None,
            "status": "no_itad_id",
            "records_downloaded": 0,
            "elapsed_seconds": 0,
            "extracted_at": datetime.now(timezone.utc).isoformat(),
            "error": None,
        })
        continue

    itad_id = str(row.itad_id)
    json_path = HISTORY_DIR / f"history_{appid}.json"

    # Reutilizar únicamente extracciones marcadas como exitosas.
    previous = manifest_by_appid.get(appid, {})
    """if (
        previous.get("status") == "success"
        and json_path.exists()
    ):
        with json_path.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        all_records.extend(
            normalize_history(appid, itad_id, payload)
        )
        continue"""

    if (
        previous.get("status") == "success"
        and json_path.exists()
    ):
        continue

    started = time.perf_counter()
    rate_limit_detected = False

    try:
        history = fetch_history(itad_id, appid)

        # Guardado atómico del JSON original.
        temp_path = json_path.with_suffix(".tmp")

        with temp_path.open("w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)

        temp_path.replace(json_path)

        normalized = normalize_history(appid, itad_id, history)
        all_records.extend(normalized)

        new_manifest_rows.append({
            "appid": appid,
            "itad_id": itad_id,
            "status": "success",
            "records_downloaded": len(history),
            "elapsed_seconds": round(
                time.perf_counter() - started, 3
            ),
            "extracted_at": datetime.now(timezone.utc).isoformat(),
            "error": None,
        })

    except ITADRateLimitError as exc:

        rate_limit_detected = True

        new_manifest_rows.append({
            "appid": appid,
            "itad_id": itad_id,
            "status": "error",
            "records_downloaded": 0,
            "elapsed_seconds": round(
                time.perf_counter() - started, 3
            ),
            "extracted_at": datetime.now(timezone.utc).isoformat(),
            "error": str(exc),
        })

    except Exception as exc:

        new_manifest_rows.append({
            "appid": appid,
            "itad_id": itad_id,
            "status": "error",
            "records_downloaded": 0,
            "elapsed_seconds": round(
                time.perf_counter() - started, 3
            ),
            "extracted_at": datetime.now(timezone.utc).isoformat(),
            "error": str(exc),
        })

    # Guardar progreso.
    current_manifest = pd.DataFrame(
        list(manifest_by_appid.values()) + new_manifest_rows
    )

    current_manifest = current_manifest.drop_duplicates(
        subset=["appid"], keep="last"
    )

    current_manifest.to_csv(HISTORY_MANIFEST_PATH, index=False)

    last_result = new_manifest_rows[-1]

    print(
        f"Procesado appid={appid} | "
        f"estado={last_result['status']} | "
        f"registros={last_result['records_downloaded']}"
    )

    # Si persiste el límite, detener toda la extracción.
    if rate_limit_detected:
        print(
            "\nITAD sigue devolviendo HTTP 429. "
            "Se detiene la extracción para evitar más peticiones."
        )
        stop_extraction = True
        break

    time.sleep(2)

In [ ]:
# comprobamos el estado de las ejecuciones

df_history_manifest = pd.read_csv(HISTORY_MANIFEST_PATH)

print("Estados de extracción:")
display(
    df_history_manifest["status"].value_counts()
)

print("\nÚltimos 10 registros:")
display(
    df_history_manifest.tail(10)
)

print("\nÚltimos errores:")
display(
    df_history_manifest.loc[
        df_history_manifest["status"] == "error",
        ["appid", "itad_id", "error"]
    ].tail(10)
)

In [ ]:
success_count = (
    df_history_manifest["status"] == "success"
).sum()

print(f"AppID exitosos: {success_count:,}")

In [ ]:
df_itad_history = pd.DataFrame(all_records)

if not df_itad_history.empty:
    df_itad_history["timestamp"] = pd.to_datetime(
        df_itad_history["timestamp"], errors="coerce", utc=True
    )

    df_itad_history = df_itad_history.drop_duplicates(
        subset=["appid", "timestamp", "shop_id"],
        keep="last",
    )

df_itad_history.to_csv(HISTORY_CSV_PATH, index=False)

print("Juegos procesados:", len(df_mapping))
print("Filas de historial:", len(df_itad_history))
print("CSV:", HISTORY_CSV_PATH)
print("Manifiesto:", HISTORY_MANIFEST_PATH)

In [ ]:
manifest = pd.read_csv(HISTORY_MANIFEST_PATH)

print("Estados de extracción:")
display(manifest["status"].value_counts(dropna=False).to_frame("juegos"))

print("\nRegistros históricos:")
print("Total de filas:", len(df_itad_history))
print("Juegos con historial:", df_itad_history["appid"].nunique())
print(
    "Juegos sin registros:",
    manifest.loc[
        manifest["status"].eq("success")
        & manifest["records_downloaded"].eq(0),
        "appid"
    ].nunique()
)

print("\nCalidad:")
print(
    "Fechas nulas:",
    df_itad_history["timestamp"].isna().sum()
)
print(
    "Precios nulos:",
    df_itad_history["price_amount"].isna().sum()
)
print(
    "Monedas:",
    df_itad_history["price_currency"].value_counts(dropna=False).to_dict()
)
print(
    "Duplicados por juego, fecha y tienda:",
    df_itad_history.duplicated(
        ["appid", "timestamp", "shop_id"]
    ).sum()
)

display(
    df_itad_history.sort_values("timestamp").head(10)
)

In [ ]:
# 4. Calidad de los datos
print(
    "Timestamps nulos:",
    df_itad_history["timestamp"].isna().sum()
)
print(
    "Precios nulos:",
    df_itad_history["price_amount"].isna().sum()
)
print(
    "Duplicados por juego, fecha y tienda:",
    df_itad_history.duplicated(
        subset=["appid", "timestamp", "shop_id"]
    ).sum()
)

In [ ]:
# 1. Estados del manifiesto: comprobar los 500 juegos
display(
    manifest["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .to_frame("juegos")
)

# 2. Distribución de monedas
display(
    df_itad_history["price_currency"]
    .value_counts(dropna=False)
    .rename_axis("currency")
    .to_frame("observaciones")
)

# 3. Tiendas asociadas a cada moneda
currency_shop = (
    df_itad_history
    .groupby(["price_currency", "shop_name"], dropna=False)
    .agg(
        observations=("appid", "size"),
        games=("appid", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["price_currency", "observations"],
        ascending=[True, False],
    )
)

display(currency_shop)

# 4. Comprobar si los juegos tienen varias monedas
currencies_per_game = (
    df_itad_history
    .groupby("appid")["price_currency"]
    .nunique()
    .value_counts()
    .sort_index()
)

display(
    currencies_per_game.rename_axis(
        "num_currencies_per_game"
    ).to_frame("games")
)

# 5. Inspeccionar ejemplos de registros que no están en EUR
display(
    df_itad_history.loc[
        df_itad_history["price_currency"].ne("EUR"),
        [
            "appid",
            "timestamp",
            "shop_id",
            "shop_name",
            "price_amount",
            "price_currency",
        ],
    ].head(10)
)

In [ ]:
# Revisar los motivos de error
display(
    manifest.loc[
        manifest["status"].eq("error")
    ].head(20)
)

In [ ]:
display(
    manifest.loc[manifest["status"].eq("error")]
    .filter(regex="status|error|message|appid|attempt")
    .head(30)
)

In [ ]:
# Monedas por tienda
currency_shop = (
    df_itad_history
    .groupby(["price_currency", "shop_name"], dropna=False)
    .agg(
        observations=("appid", "size"),
        games=("appid", "nunique")
    )
    .reset_index()
    .sort_values(
        ["price_currency", "observations"],
        ascending=[True, False]
    )
)

display(currency_shop)

In [ ]:
cols = [
    "appid",
    "timestamp",
    "shop_id",
    "shop_name",
    "price_amount",
    "regular_amount",
    "price_currency",
    "discount_pct"
]

display(
    df_itad_history.loc[
        df_itad_history["price_currency"].isin(["USD", "GBP"]),
        cols
    ].head(30)
)

In [ ]:
print("IGDB:", df_igdb.columns.tolist())
print("ITAD:", df_itad_mapping.columns.tolist())
print("Muestra Steam:", df_steam_sample.columns.tolist())


In [ ]:
df_steam_sample = df_steam_sample.rename(columns={"id": "igdb_id"})
df_steam_sample.head()

In [ ]:
# 1. Base de la muestra: una fila por appid.
df_master = df_steam_sample[
    ["appid", "igdb_id"]
].copy()

df_master["appid"] = pd.to_numeric(
    df_master["appid"], errors="coerce"
)

df_master = df_master.dropna(subset=["appid"]).copy()
df_master["appid"] = df_master["appid"].astype("int64")

# Comprobar que cada appid tiene una única correspondencia IGDB.
conflicts = (
    df_master.groupby("appid")["igdb_id"]
    .nunique(dropna=True)
)

if conflicts.gt(1).any():
    raise ValueError(
        "Hay appid asociados a varios juegos IGDB. "
        "Revisa la tabla de correspondencias antes de continuar."
    )

df_master = df_master.drop_duplicates(subset=["appid"])

# 2. Añadir los metadatos de IGDB.
df_igdb_master = df_igdb.copy()

if "id" in df_igdb_master.columns:
    df_igdb_master = df_igdb_master.rename(
        columns={"id": "igdb_id"}
    )

if df_igdb_master["igdb_id"].duplicated().any():
    raise ValueError(
        "La tabla IGDB contiene IDs duplicados. "
        "Revisa la extracción antes de hacer el merge."
    )

df_master = df_master.merge(
    df_igdb_master,
    on="igdb_id",
    how="left",
    validate="many_to_one",
    indicator="_igdb_match",
)

print("Juegos en la tabla maestra:", len(df_master))
print("Con metadatos IGDB:", df_master["_igdb_match"].eq("both").sum())
print("Sin metadatos IGDB:", df_master["_igdb_match"].eq("left_only").sum())

df_master = df_master.drop(columns="_igdb_match")

# 3. Añadir la correspondencia ITAD.
df_itad_master = df_itad_mapping[
    ["appid", "itad_id"]
].copy()

df_itad_master["appid"] = pd.to_numeric(
    df_itad_master["appid"], errors="coerce"
)
df_itad_master = df_itad_master.dropna(subset=["appid"])
df_itad_master["appid"] = df_itad_master["appid"].astype("int64")

if df_itad_master["appid"].duplicated().any():
    raise ValueError(
        "La correspondencia ITAD contiene appid duplicados. "
        "Comprueba si tienen IDs ITAD diferentes."
    )

df_master = df_master.merge(
    df_itad_master,
    on="appid",
    how="left",
    validate="one_to_one",
)

# 4. Validaciones finales.
assert len(df_master) == 500
assert df_master["appid"].is_unique

display(df_master.head())
print("Dimensiones:", df_master.shape)

In [ ]:
# Limpiar y preparar las fechas

df_reviews = df_steam_reviews.copy()

# Normalizar tipos.
df_reviews["appid"] = pd.to_numeric(
    df_reviews["appid"], errors="coerce"
).astype("Int64")

# Convertir timestamps Unix a fechas UTC.
df_reviews["timestamp_created"] = pd.to_numeric(
    df_reviews["timestamp_created"],
    errors="coerce"
)

df_reviews["timestamp_updated"] = pd.to_numeric(
    df_reviews["timestamp_updated"],
    errors="coerce"
)

df_reviews["review_date"] = pd.to_datetime(
    df_reviews["timestamp_created"],
    unit="ns",
    utc=True,
    errors="coerce"
)

# Crear periodo mensual.
df_reviews["review_month"] = (
    df_reviews["review_date"].dt.strftime("%Y-%m")
)

"""df_reviews["review_month"] = (
    df_reviews["review_date"].dt.to_period("M")
)"""

# Asegurar que la valoración sea booleana.
df_reviews["voted_up"] = df_reviews["voted_up"].astype("boolean")

print("Total de reseñas:", len(df_reviews))
print("Juegos con reseñas:", df_reviews["appid"].nunique())
print("Fechas nulas:", df_reviews["review_date"].isna().sum())
print("Valoraciones nulas:", df_reviews["voted_up"].isna().sum())
print(
    "IDs de reseña duplicados:",
    df_reviews["recommendationid"].duplicated().sum()
)

In [ ]:
df_reviews.isna().sum()

In [ ]:
# Convertir timestamps Unix a fechas UTC.
df_reviews["timestamp_created"] = pd.to_numeric(
    df_reviews["timestamp_created"],
    errors="coerce"
)

df_reviews["timestamp_updated"] = pd.to_numeric(
    df_reviews["timestamp_updated"],
    errors="coerce"
)

df_reviews["review_date"] = pd.to_datetime(
    df_reviews["timestamp_created"],
    unit="ns",
    utc=True,
    errors="coerce"
)

# Crear periodo mensual.
df_reviews["review_month"] = (
    df_reviews["review_date"].dt.strftime("%Y-%m")
)

# Comprobar resultados.
print("Fechas nulas:", df_reviews["review_date"].isna().sum())

display(
    df_reviews[
        ["timestamp_created", "review_date", "review_month"]
    ].head(10)
)

In [ ]:
df_reviews_valid = df_reviews.dropna(
    subset=["appid", "review_date", "voted_up"]
).copy()

df_reviews_monthly = (
    df_reviews_valid
    .groupby(["appid", "review_month"], observed=True)
    .agg(
        review_count=("recommendationid", "count"),
        positive_share=("voted_up", "mean"),
        avg_playtime_at_review=(
            "author.playtime_at_review", "mean"
        ),
    )
    .reset_index()
)

df_reviews_monthly["positive_pct"] = (
    df_reviews_monthly["positive_share"] * 100
)

display(df_reviews_monthly.sample(10))

In [ ]:
print("Fechas nulas:", df_reviews["review_date"].isna().sum())
print("Primera reseña:", df_reviews["review_date"].min())
print("Última reseña:", df_reviews["review_date"].max())
print("Total filas mensuales:", len(df_reviews_monthly))
print("Juegos:", df_reviews_monthly["appid"].nunique())

display(df_reviews_monthly.head(10))

# ESA

Pregunta de negocio	+ Fuentes

¿Cómo ha evolucionado el mercado de videojuegos?	ESA + Newzoo

¿Qué plataformas concentran más ingresos y jugadores?	Newzoo

¿Cómo ha cambiado el perfil demográfico de los jugadores?	ESA

¿Existe relación entre descuentos y volumen de reseñas?	Steam + ITAD

¿Los juegos con mejores valoraciones mantienen mayor actividad?	IGDB + Steam

¿Existe relación entre popularidad en streaming y actividad de los jugadores?	Twitch + Steam

In [ ]:
# Directorios
BASE_DIR = Path("data/raw/esa")

HTML_DIR = BASE_DIR / "html"
PDF_DIR = BASE_DIR / "pdf"
METADATA_DIR = BASE_DIR / "metadata"

for folder in [HTML_DIR, PDF_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# URL oficial
ESA_URL = (
    "https://www.theesa.com/resources/"
    "essential-facts-about-the-us-video-game-industry/"
    "2026-data/"
)

# Petición HTTP
headers = {
    "User-Agent": "Mozilla/5.0 (compatible; VideoGamesDataAnalysis/1.0)"
}

response = requests.get(
    ESA_URL,
    headers=headers,
    timeout=30
)

response.raise_for_status()

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

# Guardar HTML original
html_path = HTML_DIR / "esa_essential_facts_2026.html"

html_path.write_text(
    response.text,
    encoding="utf-8"
)

print("HTML guardado:", html_path)

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

# Eliminar elementos que no aportan contenido al análisis
for element in soup([
    "script",
    "style",
    "nav",
    "footer",
    "header"
]):
    element.decompose()

# Extraer texto
page_text = soup.get_text(
    separator="\n",
    strip=True
)

# Guardar texto
text_path = HTML_DIR / "esa_essential_facts_2026.txt"

text_path.write_text(
    page_text,
    encoding="utf-8"
)

print(page_text[:5000])

In [ ]:
ESA_PDF_URL = (
    "https://www.theesa.com/wp-content/uploads/2026/06/"
    "2026-Essential-Facts-Booklet-05-27-26.pdf"
)

pdf_path = PDF_DIR / "esa_essential_facts_2026.pdf"

with requests.get(
    ESA_PDF_URL,
    headers=headers,
    stream=True,
    timeout=120
) as response:

    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "")

    if "pdf" not in content_type.lower():
        raise ValueError(
            f"El recurso descargado no parece un PDF: {content_type}"
        )

    with open(pdf_path, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

print("PDF descargado:", pdf_path)
print("Tamaño:", round(pdf_path.stat().st_size / (1024 * 1024), 2), "MB")